# E1 · Did the GENIUS Act move stablecoin volatility? (exploratory, work in progress)

Hourly realised variance from 5-minute log returns on **active minutes only**, deseasonalised by hour-of-week, with
a jump regression ($y = a + \beta\,\text{post} + \gamma\,\text{trend}$, Newey–West) around each legislative
milestone, plus placebo inference at random same-hour-of-week dates.

> **Status.** Exploratory extension, not part of the submitted paper. Inputs (2025 one-minute Binance.US
> USDT/USD and USDC/USD, Binance USDC/USDT, ~150 MB) are not committed; download them with
> `stablecoin_risk.data.collectors.fetch_binance_us_bulk(..., interval="1m")` into `data/external/genius_2025/`.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import config, event_study as es
from stablecoin_risk.data import loaders

d = config.EXTERNAL_DIR / "genius_2025"
files = {"USDT/USD (Binance.US)": "binanceus_USDTUSD_1m_20250101_20260101.csv",
         "USDC/USD (Binance.US)": "binanceus_USDCUSD_1m_20250101_20260101.csv",
         "USDC/USDT (Binance)": "binance_USDC_USDT_1min_2025.csv"}
rv = {}
for name, f in files.items():
    price = es.active_prices(loaders.read_ohlcv_csv(d / f, window=None))
    rv[name] = es.realized_variance(price)
    print(f"{name:24s} active minutes {price.notna().mean():6.1%}   usable hours {rv[name]['log_rv'].notna().mean():6.1%}")

USDT/USD (Binance.US)    active minutes  65.7%   usable hours  89.2%


USDC/USD (Binance.US)    active minutes   3.5%   usable hours   0.8%


USDC/USDT (Binance)      active minutes  16.4%   usable hours  13.5%


Only USDT/USD trades often enough for hourly realised variance (89% of hours usable). USDC/USD on Binance.US
prints a trade in 3.5% of minutes and USDC/USDT in 16%, too sparse for this design, so the study focuses on
USDT/USD.

In [3]:
res = es.run_event_study(rv["USDT/USD (Binance.US)"]["log_rv"], n_placebo=200)
res.round(3)

,event,window_h,n,beta_post,rv_change_pct,hac_p,placebo_p,n_placebo
0,Senate passed,6,12,-0.54,-41.73,0.445,0.643,42
1,Senate passed,12,24,-1.053,-65.12,0.014,0.311,45
2,Senate passed,24,48,-0.347,-29.35,0.357,0.771,48
3,Senate passed,48,96,-1.348,-74.02,0.031,0.188,48
4,House passed,6,11,2.479,"1,093",0.022,0.14,43
5,House passed,12,23,2.805,"1,552",0.012,0,43
6,House passed,24,47,1.083,195.5,0.259,0.17,47
7,House passed,48,95,0.404,49.71,0.723,0.688,48
8,Signed into law,6,12,-0.465,-37.18,0.22,0.718,39
9,Signed into law,12,24,1.028,179.5,0.135,0.262,42


**Interpretation (tentative).** USDT/USD realised volatility jumps after the House vote (12-hour window: HAC
p = 0.01, placebo p < 0.01) and around the signing (24-hour window: placebo p < 0.01). The House vote and the
signing are **four hours apart**, so their windows overlap and the two events cannot be separated. The Senate vote
is followed by lower volatility in some windows, but that does not survive placebo inference (p ≥ 0.19).
Results also vary with window length, so specification search is a real risk. Next steps: a joint
House-and-signing event, a second USDT venue (Coinbase/Kraken), and a pre-registered window.